In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.features.plate_discipline import discipline_profile
from src.features.batted_ball import quality_profile
from src.features.sample_size import STABILIZATION, is_reliable
from src.data.player_ids import load_player_ids, display_name

df = load_all_snapshots()

rows = []
for bid, g in df.groupby("batter"):
    if len(g) < 500:
        continue
    p = discipline_profile(g)
    p.update(quality_profile(g))
    p["batter"] = bid
    rows.append(p)

prof = pd.DataFrame(rows).set_index("batter")
ids = load_player_ids(prof.index.tolist())
prof = prof.join(display_name(ids))
print(f"{len(prof)} batters")

425 batters


In [2]:
# Three axes, chosen from the Day 16 correlation structure.
# Barrel% is used for power; HardHit% is excluded (r = 0.78, redundant).
AXES = {
    "discipline": ["chase_pct", "zone_swing_pct"],
    "contact":    ["zone_contact_pct"],
    "power":      ["barrel_pct"],
}

# Direction: is higher better?
HIGHER_IS_BETTER = {
    "chase_pct": False,        # chasing is bad
    "zone_swing_pct": True,
    "zone_contact_pct": True,
    "barrel_pct": True,
}

def z_score(series, higher_better=True):
    z = (series - series.mean()) / series.std()
    return z if higher_better else -z

qualified = prof[
    (prof["n_out_of_zone"] >= STABILIZATION["chase_pct"][1])
    & (prof["n_zone_swings"] >= STABILIZATION["zone_contact_pct"][1])
    & (prof["bbe"] >= STABILIZATION["barrel_pct"][1])
].copy()

print(f"{len(qualified)} batters clear every stabilization threshold")
print(f"({len(prof) - len(qualified)} dropped)")

for axis, metrics in AXES.items():
    parts = [z_score(qualified[m], HIGHER_IS_BETTER[m]) for m in metrics]
    qualified[f"z_{axis}"] = sum(parts) / len(parts)

print()
print(qualified[[f"z_{a}" for a in AXES]].describe().round(3).to_string())
print()
print(qualified[[f"z_{a}" for a in AXES]].corr().round(3).to_string())

343 batters clear every stabilization threshold
(82 dropped)

       z_discipline  z_contact  z_power
count       343.000    343.000  343.000
mean          0.000     -0.000   -0.000
std           0.483      1.000    1.000
min          -1.505     -3.271   -1.781
25%          -0.330     -0.647   -0.731
50%           0.046      0.064   -0.114
75%           0.341      0.695    0.638
max           1.192      2.390    4.802

              z_discipline  z_contact  z_power
z_discipline         1.000     -0.270    0.263
z_contact           -0.270      1.000   -0.535
z_power              0.263     -0.535    1.000


In [3]:
# Equal weights as the starting point. Any other choice needs a reason.
qualified["score_equal"] = qualified[[f"z_{a}" for a in AXES]].mean(axis=1)

top = qualified.nlargest(15, "score_equal")
cols = ["name", "score_equal", "z_discipline", "z_contact", "z_power",
        "chase_pct", "zone_contact_pct", "barrel_pct"]
print(top[cols].round(3).to_string(index=False))

              name  score_equal  z_discipline  z_contact  z_power  chase_pct  zone_contact_pct  barrel_pct
      Judge, Aaron        1.498         0.915     -1.225    4.802      0.179             0.796       0.270
     Seager, Corey        1.243         1.192      0.659    1.880      0.267             0.884       0.153
        Soto, Juan        1.069         0.172      0.036    3.000      0.178             0.855       0.198
      Tucker, Kyle        1.005         1.024      0.714    1.276      0.171             0.887       0.129
    Ohtani, Shohei        0.940         0.392     -1.075    3.503      0.259             0.803       0.218
   Álvarez, Yordan        0.884        -0.049      1.014    1.687      0.291             0.901       0.145
       Witt, Bobby        0.759         0.145      0.500    1.632      0.315             0.877       0.143
      Marte, Ketel        0.749         0.582      0.532    1.133      0.253             0.878       0.123
 Walker, Christian        0.706      

In [4]:
# Sensitivity: does the ranking survive different weightings?
weight_sets = {
    "equal":            {"discipline": 1/3, "contact": 1/3, "power": 1/3},
    "power_heavy":      {"discipline": 0.2, "contact": 0.2, "power": 0.6},
    "discipline_heavy": {"discipline": 0.6, "contact": 0.2, "power": 0.2},
    "contact_heavy":    {"discipline": 0.2, "contact": 0.6, "power": 0.2},
}

for name, w in weight_sets.items():
    qualified[f"s_{name}"] = sum(qualified[f"z_{a}"] * w[a] for a in AXES)

score_cols = [f"s_{n}" for n in weight_sets]
ranks = qualified[score_cols].rank(ascending=False)

print("rank correlation between weightings:")
print(ranks.corr(method="spearman").round(3).to_string())
print()
print("top 10 under each weighting:")
for c in score_cols:
    names = qualified.nlargest(10, c)["name"].tolist()
    print(f"  {c:20s} {', '.join(n.split(',')[0] for n in names)}")

rank correlation between weightings:
                    s_equal  s_power_heavy  s_discipline_heavy  s_contact_heavy
s_equal               1.000          0.740               0.841            0.629
s_power_heavy         0.740          1.000               0.656            0.053
s_discipline_heavy    0.841          0.656               1.000            0.347
s_contact_heavy       0.629          0.053               0.347            1.000

top 10 under each weighting:
  s_equal              Judge, Seager, Soto, Tucker, Ohtani, Álvarez, Witt, Marte, Walker, Grichuk
  s_power_heavy        Judge, Ohtani, Soto, Stanton, Seager, Carpenter, Rooker, Álvarez, Toglia, Tucker
  s_discipline_heavy   Judge, Seager, Tucker, Walker, Ohtani, Semien, Soto, Marte, Tatís, Merrill
  s_contact_heavy      Kwan, Betts, Pasquantino, Seager, Álvarez, Lemahieu, Flores, Arráez, Grichuk, Ramírez
